# Task 5: Build Regression Models (Method #6 Core Analysis)

Two parallel OLS regressions -- one predicting `log_likes`, one predicting `log_comments` --
using the rule-based (Task 3) and zero-shot (Task 4) features, with `trend` as a control.

**Note on scope:** Task 4 used a stratified sample (9,423 posts: sourdough/banana_bread/
dalgona_coffee capped at 3,000 each, baked_oats/feta_pasta kept full), so this regression runs
on that same sample, not the full 126,410-row English-only dataset.

**Baseline:** sourdough is the fixed reference trend for both models (settled decision, not
dependent on Task 2's engagement ranking -- see `CLAUDE_CODE_TASKLIST.md`).

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

## Step 1: Load Task 3+4's combined output

In [2]:
combined = pd.read_csv("../../output/cleaned_data/trends_combined_english_features.csv")
print(combined.shape)

(9423, 37)


## Step 2: Dummy-encode `trend`, sourdough as baseline

In [3]:
trend_dummies = pd.get_dummies(combined["trend"], prefix="trend", drop_first=False)
trend_dummies = trend_dummies.drop(columns=["trend_sourdough"])
print(trend_dummies.columns.tolist())

['trend_baked_oats', 'trend_banana_bread', 'trend_dalgona_coffee', 'trend_feta_pasta']


## Step 3: Assemble the predictor matrix

In [4]:
rule_based_cols = ["log_word_count", "sentiment_score", "hashtag_count",
                    "exclamation_count", "question_count", "emoji_count"]
zero_shot_cols = ["is_recipe_instructional", "is_personal_lifestyle", "is_media_repost",
                   "is_meme_joke", "is_spam_low_content"]

predictor_matrix = pd.concat([
    combined[rule_based_cols],
    combined["is_covid_framed"].astype(int).rename("is_covid_framed"),
    combined[zero_shot_cols],
    trend_dummies,
], axis=1)
print(predictor_matrix.shape)
print(predictor_matrix.columns.tolist())

(9423, 16)
['log_word_count', 'sentiment_score', 'hashtag_count', 'exclamation_count', 'question_count', 'emoji_count', 'is_covid_framed', 'is_recipe_instructional', 'is_personal_lifestyle', 'is_media_repost', 'is_meme_joke', 'is_spam_low_content', 'trend_baked_oats', 'trend_banana_bread', 'trend_dalgona_coffee', 'trend_feta_pasta']


## Step 4: Fit the likes model

In [5]:
X = sm.add_constant(predictor_matrix)
model_likes = sm.OLS(combined["log_likes"], X.astype(float)).fit()
print(model_likes.summary())

                            OLS Regression Results                            
Dep. Variable:              log_likes   R-squared:                       0.087
Model:                            OLS   Adj. R-squared:                  0.085
Method:                 Least Squares   F-statistic:                     55.77
Date:                Sun, 23 Aug 2026   Prob (F-statistic):          2.96e-171
Time:                        21:44:20   Log-Likelihood:                -17529.
No. Observations:                9423   AIC:                         3.509e+04
Df Residuals:                    9406   BIC:                         3.521e+04
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

## Step 5: Fit the comments model

**Caveat:** comments have confirmed zero-inflation (unlike likes' clean log-normal shape, see
Task 2 / `MASTER_PROJECT_REFERENCE.md`), so this model's fit should be treated as less reliable
than the likes model -- stated explicitly here, not presented with equal confidence.

In [6]:
model_comments = sm.OLS(combined["log_comments"], X.astype(float)).fit()
print(model_comments.summary())

                            OLS Regression Results                            
Dep. Variable:           log_comments   R-squared:                       0.177
Model:                            OLS   Adj. R-squared:                  0.175
Method:                 Least Squares   F-statistic:                     126.0
Date:                Sun, 23 Aug 2026   Prob (F-statistic):               0.00
Time:                        21:44:20   Log-Likelihood:                -16630.
No. Observations:                9423   AIC:                         3.329e+04
Df Residuals:                    9406   BIC:                         3.341e+04
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

## Step 6: Coefficients, p-values, and R² side by side

In [7]:
comparison = pd.DataFrame({
    "coef_likes": model_likes.params,
    "pval_likes": model_likes.pvalues,
    "coef_comments": model_comments.params,
    "pval_comments": model_comments.pvalues,
})
comparison["sig_likes"] = comparison["pval_likes"] < 0.05
comparison["sig_comments"] = comparison["pval_comments"] < 0.05

print(comparison.round(4))
print()
print(f"Likes model:    R2 = {model_likes.rsquared:.4f}, Adj R2 = {model_likes.rsquared_adj:.4f}, N = {int(model_likes.nobs)}")
print(f"Comments model: R2 = {model_comments.rsquared:.4f}, Adj R2 = {model_comments.rsquared_adj:.4f}, N = {int(model_comments.nobs)}")

                         coef_likes  pval_likes  coef_comments  pval_comments  \
const                        4.5521      0.0000         1.0160         0.0000   
log_word_count               0.0764      0.0033         0.1066         0.0000   
sentiment_score              0.0153      0.7496         0.1037         0.0177   
hashtag_count                0.0190      0.0000         0.0225         0.0000   
exclamation_count            0.0328      0.0002         0.0611         0.0000   
question_count               0.0755      0.0002         0.1003         0.0000   
emoji_count                  0.0195      0.0000         0.0240         0.0000   
is_covid_framed             -0.1253      0.0045        -0.0372         0.3535   
is_recipe_instructional      0.4911      0.0000         0.5581         0.0000   
is_personal_lifestyle        0.1446      0.0003         0.4544         0.0000   
is_media_repost              0.3197      0.0000        -0.1556         0.0002   
is_meme_joke                

Low R² here is expected, not a failure -- individual post virality is inherently hard to
predict from a handful of text features; the value of this analysis is in which features show
significant, interpretable effects, not in raw predictive accuracy.

## Step 7: Multicollinearity check (VIF) across the full predictor set

In [8]:
vif_data = pd.DataFrame()
vif_data["feature"] = predictor_matrix.columns
vif_data["VIF"] = [
    variance_inflation_factor(predictor_matrix.astype(float).values, i)
    for i in range(predictor_matrix.shape[1])
]
print(vif_data.sort_values("VIF", ascending=False).to_string(index=False))

                feature       VIF
         log_word_count 12.352340
        sentiment_score  5.241210
          hashtag_count  4.266553
     trend_banana_bread  2.070349
   trend_dalgona_coffee  2.048592
  is_personal_lifestyle  1.972195
is_recipe_instructional  1.844434
      exclamation_count  1.703514
            emoji_count  1.574671
    is_spam_low_content  1.519693
        is_media_repost  1.418748
        is_covid_framed  1.315507
         question_count  1.302567
           is_meme_joke  1.106327
       trend_feta_pasta  1.104555
       trend_baked_oats  1.072939


VIF > 5-10 is the conventional concern threshold; report here, not a blocker for this
project's scope per the task list.